# Foundation of Computer science 2026-Project
Author: ***Askar Amro Emad Hassan (946807)***

## Use of LLMs / Generative AI

All analysis, logic, and code in this project were developed and implemented independently by the author. Generative AI was used solely to assist with notebook formatting, Markdown organization, and improving the readability of comments and outputs.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np

In [2]:
# Reading the CSV file
student_dropout_risk = pd.read_csv("data/student_dropout_risk_2026.csv.gz")
student_dropout_risk.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


# Q1
Extract all students that study Online, have an attendance_rate smaller than 50, and have more than 5 missed_deadlines. Store them in a dataframe called at_risk.

In [3]:
at_risk = student_dropout_risk[(student_dropout_risk["study_mode"] == "Online") &
                     (student_dropout_risk["attendance_rate"] < 50) &
                     (student_dropout_risk["missed_deadlines"] > 5)]
at_risk.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,2,6.6,0,7,7,0,5,11,5.4,1
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,3,7.8,0,17,5,0,4,6,8.1,1


# Q2
Then compute, for each course_type, how many of those students there are and what fraction of them has dropped out. Store the result in a dataframe called at_risk_by_course, with columns students and dropout_rate, sorted by decreasing dropout_rate.

In [4]:
at_risk_by_course = at_risk.groupby("course_type").agg(students=("course_type", "size"),
                                   dropout_rate=("dropout", "mean")).sort_values("dropout_rate", ascending=False)
at_risk_by_course

,students,dropout_rate
course_type,,
Law & Social Sciences,29,0.896552
STEM,47,0.765957
Business,47,0.765957
Arts & Humanities,40,0.750000
Health Sciences,34,0.647059


# Q3
Add to the dataframe the following columns:

- pass_rate: the ratio between units_passed and units_enrolled;
- gpa_change: the difference between gpa_semester_2 and gpa_semester_1;
- gpa_trend: a string that is "improving" if gpa_change is larger than 0.5, "declining" if gpa_change is smaller than -0.5, and "stable" otherwise.

In [5]:
student_dropout_risk["pass_rate"] = student_dropout_risk["units_passed"] / student_dropout_risk["units_enrolled"]
student_dropout_risk["gpa_change"] = student_dropout_risk["gpa_semester_2"] - student_dropout_risk["gpa_semester_1"]
student_dropout_risk["gpa_trend"] = "stable"
student_dropout_risk.loc[student_dropout_risk["gpa_change"] > 0.5, "gpa_trend"] = "improving"
student_dropout_risk.loc[student_dropout_risk["gpa_change"] < -0.5, "gpa_trend"] = "declining"

student_dropout_risk[["pass_rate", "gpa_change", "gpa_trend"]].head()

,pass_rate,gpa_change,gpa_trend
0,1.000000,-0.89,declining
1,0.785714,-0.03,stable
2,0.823529,-0.70,declining
3,0.846154,-0.41,stable
4,0.875000,-1.05,declining


# Q4
Verify that, for every student, units_passed + units_failed equals units_enrolled; if not, report the rows violating this condition.

In [6]:
# NO duplicated students
student_dropout_risk["person_id"].is_unique

True

In [7]:
# The statement is correct
statement = (student_dropout_risk["units_enrolled"] == student_dropout_risk["units_passed"] +
                                                student_dropout_risk["units_failed"])
print(f"The statement is {statement.all()}")

# In case of violation
# violating_rows = student_dropout_risk[~statement]
# violating_rows

The statement is True


# Q5
Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop.

In [8]:
student_dropout_risk.groupby("gpa_trend").agg(dropout_rate=("dropout", "mean"))

,dropout_rate
gpa_trend,
declining,0.265086
improving,0.589158
stable,0.391231


In [9]:
# keep="all" returns duplicated values in case more than a student has the same gpa_change
largest_gpa_drop = student_dropout_risk.nsmallest(10, "gpa_change", keep="all")[["person_id"]]
largest_gpa_drop

,person_id
23225,23226
96224,96225
88489,88490
51824,51825
58935,58936
46714,46715
43035,43036
72663,72664
76705,76706
23427,23428


# Q6
Partition the students into the following groups, according to weekly_work_hours: 0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes).

In [10]:
# To make sure that weekly_work_hours column is integer and doesn't have any decimal values
student_dropout_risk["weekly_work_hours"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 100000 entries, 0 to 99999
Series name: weekly_work_hours
Non-Null Count   Dtype
--------------   -----
100000 non-null  int64
dtypes: int64(1)
memory usage: 781.4 KB


In [11]:
student_dropout_risk["weekly_work_hours_group"] = pd.cut(student_dropout_risk["weekly_work_hours"],
                                                         bins=[-np.inf, 0, 10, 20, 30, np.inf],
                                                         labels=["0", "1-10", "11-20", "21-30", "more than 30"],
                                                         right=True)  # to include the rightmost edge
student_dropout_risk[["weekly_work_hours", "weekly_work_hours_group"]].head()

,weekly_work_hours,weekly_work_hours_group
0,24,21-30
1,0,0
2,16,11-20
3,0,0
4,0,0


# Q7
Build a risk score by summing:

- 2 points if the tuition is not up to date;
- 1 point if stress_level is at least 8;
- 1 point if sleep_hours is smaller than 5;
- 1 point if the student has a part-time job with more than 20 weekly work hours;
- 2 points if attendance_rate is smaller than 60;
- 1 point if missed_deadlines is larger than 3.

For each value of the risk score, compute the number of students and the dropout rate.

In [12]:
def risk(row):
    total = 0
    if row["tuition_up_to_date"] == 0:
        total += 2
    if row["stress_level"] >= 8:
        total += 1
    if row["sleep_hours"] < 5:
        total += 1
    if (row["part_time_job"] == 1) & (row["weekly_work_hours"] > 20):
        total += 1
    if row["attendance_rate"] < 60:
        total += 2
    if row["missed_deadlines"] > 3:
        total += 1

    return total

In [13]:
student_dropout_risk["risk_score"] = student_dropout_risk.apply(risk, axis=1)
student_dropout_risk.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend,weekly_work_hours_group,risk_score
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,0,6,37,5.7,0,1.000000,-0.89,declining,21-30,4
1,2,18,Other,USA,1,1,High,1,1,0,...,0,6,15,5.4,1,0.785714,-0.03,stable,0,2
2,3,24,Other,USA,0,0,High,0,1,1,...,0,3,4,4.3,0,0.823529,-0.70,declining,11-20,1
3,4,25,Female,USA,0,1,Middle,1,1,0,...,0,7,10,6.7,0,0.846154,-0.41,stable,0,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,10,24,5.8,0,0.875000,-1.05,declining,0,0


In [14]:
student_dropout_risk.groupby("risk_score").agg(students=("risk_score", "size"), dropout_rate=("dropout", "mean"))

,students,dropout_rate
risk_score,,
0,26955,0.152847
1,24115,0.252084
2,21984,0.337200
3,14557,0.452703
4,7613,0.577433
5,3320,0.673494
6,1149,0.794604
7,267,0.827715
8,40,0.900000


# Q8
Read the GDP per capita dataset published by Our World in Data directly from the URL

https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true
and store it in a dataframe called gdp. The column ny_gdp_pcap_pp_kd contains the GDP per capita.

Clean the dataset, removing the rows that do not correspond to a single country (e.g. World, High-income countries). For each country keep only the row with the most recent year.

In [15]:
gdp = pd.read_csv("data/gdp-per-capita-worldbank.csv")
gdp.head(10)

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
5,Afghanistan,AFG,2005,1908.1147,Asia
6,Afghanistan,AFG,2006,1929.7239,Asia
7,Afghanistan,AFG,2007,2155.3530,Asia
8,Afghanistan,AFG,2008,2191.5044,Asia
9,Afghanistan,AFG,2009,2565.0220,Asia


In [16]:
gdp["entity"].value_counts()

entity
Albania                      36
Algeria                      36
Andorra                      36
Angola                       36
Belize                       36
                             ..
Kosovo                       18
Turks and Caicos Islands     18
Faroe Islands                17
Sint Maarten (Dutch part)    17
Djibouti                     13
Name: count, Length: 212, dtype: int64

In [17]:
# East Asia and Pacific (WB)
# Europe and Central Asia (WB)
# Latin America and Caribbean (WB)
# North America (WB)
# Middle East, North Africa, Afghanistan and Pakistan (WB)
# South Asia (WB)
# Sub-Saharan Africa (WB)

# Upper-middle-income countries
# High-income countries
# Low-income countries
# Lower-middle-income countries

# World
# European Union (27)

"""
I notice that countries have value for the owid_region column so if owid_region is NaN so the row is not a country
"""

gdp = gdp[gdp["owid_region"].notna()]
gdp.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


In [18]:
gdp_recent = gdp.groupby("entity")["year"].max().reset_index()
gdp_final = gdp.merge(gdp_recent, on=["entity", "year"], how="right")
gdp_final

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2024,1964.4523,Asia
1,Albania,ALB,2025,22724.2930,Europe
2,Algeria,DZA,2025,15882.7140,Africa
3,Andorra,AND,2025,67993.0550,Europe
4,Angola,AGO,2025,8760.1900,Africa
...,...,...,...,...,...
194,Uzbekistan,UZB,2025,11622.8550,Asia
195,Vanuatu,VUT,2025,3866.9980,Oceania
196,Vietnam,VNM,2025,15457.3500,Asia
197,Zambia,ZMB,2025,3742.2563,Africa


# Q9
Join the GPD data with the students data. Compute the dropout rate for each continent.

In [19]:
# Before I join the two DFs I need to make them consistent first

In [20]:
# For consistency, I will lower case the country column in both DFs
gdp_final["entity"] = gdp_final["entity"].str.lower()

student_dropout_risk["nationality"] = student_dropout_risk["nationality"].str.lower()

In [21]:
# For consistency, I need to replace usa ==> united states and uk ==> for united kingdom
student_dropout_risk["nationality"] = student_dropout_risk["nationality"].replace({"uk": "united kingdom",
                                                                                    "usa": "united states"})

In [22]:
df = student_dropout_risk.merge(gdp_final, how="left", left_on="nationality", right_on="entity")
df.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,pass_rate,gpa_change,gpa_trend,weekly_work_hours_group,risk_score,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,1,23,Male,brazil,0,0,Low,1,0,1,...,1.000000,-0.89,declining,21-30,4,brazil,BRA,2025,20024.697,South America
1,2,18,Other,united states,1,1,High,1,1,0,...,0.785714,-0.03,stable,0,2,united states,USA,2025,76931.310,North America
2,3,24,Other,united states,0,0,High,0,1,1,...,0.823529,-0.70,declining,11-20,1,united states,USA,2025,76931.310,North America
3,4,25,Female,united states,0,1,Middle,1,1,0,...,0.846154,-0.41,stable,0,0,united states,USA,2025,76931.310,North America
4,5,15,Female,canada,0,0,Low,1,1,0,...,0.875000,-1.05,declining,0,0,canada,CAN,2025,57989.580,North America


In [23]:
df.groupby("owid_region").agg(dropout_rate=("dropout", "mean"))

,dropout_rate
owid_region,
Africa,0.316721
Asia,0.322908
Europe,0.322762
North America,0.317679
Oceania,0.321981
South America,0.317620
